In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

In [ ]:
!pip install pytorch-lightning torchmetrics tqdm pandas statsmodels -q
print(f"Working directory set to: {os.getcwd()}")

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
import pytorch_lightning as pl
import torch.nn.functional as F
import numpy as np
import os
import pandas as pd
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")



BASE_MODEL_DIR = '/content/drive/MyDrive/Colab_project/model_trained_MNIST/AlexNet_for_MNIST/model_alexnet_224/'
OUTPUT_DIR = '/content/drive/MyDrive/Colab_project/model_trained_MNIST/AlexNet_for_MNIST/results_bias_shift_alexnet_224_normalize/'

NOISE_LEVEL = 1.19   # The noise level from model testing
NUM_INSTANCES = 60
BATCH_SIZE = 128
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class LitAlexNet(pl.LightningModule):
    def __init__(self, num_classes=10):
        super(LitAlexNet, self).__init__()
        self.model = models.alexnet(weights=None)
        self.model.features[0] = nn.Conv2d(1, 64, kernel_size=11, stride=4, padding=2)
        in_features = self.model.classifier[6].in_features
        self.model.classifier[6] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)


class NoisyDataset(Dataset):
    def __init__(self, root, train=False, noise_level=0.0):
        self.base_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])
        self.resize = transforms.Resize((224, 224), antialias=True)
        self.mnist = datasets.MNIST(root=root, train=train, transform=self.base_transform, download=True)
        self.noise_level = noise_level

    def __len__(self): return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        if self.noise_level > 0:
            noise = torch.randn_like(image) * self.noise_level
            image = image + noise
        image = self.resize(image)
        return image, label

def find_checkpoint_path(base_dir, instance_num):
    filename = f"alexnet-224-{instance_num}-final.pt"
    full_path = os.path.join(base_dir, filename)
    if os.path.exists(full_path):
        return full_path
    return None


def get_distribution_stats(model, loader, num_classes=10):
    """
    Step 1: Run on Validation set to learn the Bias (S values)
    Calculates evidence using Z-Score Normalized Logits (Signal-to-Noise Ratio).
    """
    model.eval()
    evidence_stats = {i: [] for i in range(num_classes)}
    chosen_counts = np.zeros(num_classes)
    total_samples = 0

    with torch.no_grad():
        for images, _ in loader:
            images = images.to(device)
            logits = model(images)


            mean_logits = logits.mean(dim=1, keepdim=True)
            std_logits = logits.std(dim=1, keepdim=True) + 1e-8
            norm_logits = (logits - mean_logits) / std_logits

            top2 = torch.topk(norm_logits, k=2, dim=1).values
            evidence = (top2[:, 0] - top2[:, 1]).cpu().numpy()
            preds = torch.argmax(norm_logits, dim=1).cpu().numpy()

            for p, ev in zip(preds, evidence):
                chosen_counts[p] += 1
                evidence_stats[p].append(ev)
            total_samples += len(images)

    return chosen_counts, evidence_stats, total_samples

def calculate_shifts(chosen_counts, evidence_stats, total_samples, num_classes=10):
    shifts = {}
    target_prob = 1.0 / num_classes

    for i in range(num_classes):
        count = chosen_counts[i]
        if count < 10:
            shifts[i] = 0.0
            continue

        ev_values = np.array(evidence_stats[i])
        p_actual = count / total_samples

        if p_actual > target_prob:
            keep_ratio = target_prob / p_actual
            percentile_to_cut = max(0, min(100, (1 - keep_ratio) * 100))
            shifts[i] = np.percentile(ev_values, percentile_to_cut)
        else:
            shifts[i] = 0.0

    return shifts

def process_instance(instance_num):
    model_path = find_checkpoint_path(BASE_MODEL_DIR, instance_num)
    if not model_path: return

    try:
        model = LitAlexNet(num_classes=10)
        checkpoint = torch.load(model_path, map_location=device)
        if 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'], strict=False)
        else:
            model.load_state_dict(checkpoint, strict=False)
        model.to(device)
        model.eval()
    except Exception as e:
        print(f"Error loading {model_path}: {e}")
        return

    calib_dataset = NoisyDataset(root='data', train=True, noise_level=NOISE_LEVEL)
    calib_loader = DataLoader(calib_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    counts, evidence_stats, total_samples = get_distribution_stats(model, calib_loader)
    shifts = calculate_shifts(counts, evidence_stats, total_samples)

    test_dataset = NoisyDataset(root='data', train=False, noise_level=NOISE_LEVEL)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    data_rows = []

    with torch.no_grad():
        batch_idx = 0
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)

            logits = model(images)

            mean_logits = logits.mean(dim=1, keepdim=True)
            std_logits = logits.std(dim=1, keepdim=True) + 1e-8
            norm_logits = (logits - mean_logits) / std_logits

            top2 = torch.topk(norm_logits, k=2, dim=1).values
            conf_naive_batch = (top2[:, 0] - top2[:, 1]).cpu().numpy()

            preds_batch = torch.argmax(norm_logits, dim=1).cpu().numpy()
            labels_batch = labels.cpu().numpy()

            probs_batch_np = F.softmax(logits, dim=1).cpu().numpy()

            for i in range(len(labels_batch)):
                p = preds_batch[i]
                c_naive = conf_naive_batch[i]
                s = shifts[p]

                c_final = c_naive - s

                bias_metric = counts[p] / total_samples

                row = {
                    'instance': instance_num,
                    'image_index': batch_idx * BATCH_SIZE + i,
                    'true_label': labels_batch[i],
                    'response': p,
                    'accuracy': 1 if p == labels_batch[i] else 0,
                    'conf_naive': c_naive,
                    'conf_final': c_final,
                    'shift_applied': s,
                    'bias_metric': bias_metric,
                    'conf_softmax': np.max(probs_batch_np[i])
                }
                data_rows.append(row)
            batch_idx += 1

    df = pd.DataFrame(data_rows)
    save_path = os.path.join(OUTPUT_DIR, f'bias_correction_alexnet_inst{instance_num}.csv')
    df.to_csv(save_path, index=False)

if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Starting Bias Shift Generation (Noise {NOISE_LEVEL})...")
    for i in tqdm(range(NUM_INSTANCES)):
        process_instance(i)
    print("Done!")